# Run A / Run B overlap visualization

This global notebook compares two runs/configurations at the example level.  It separates examples into three sections:

1. **Both correct**: run A and run B both score the example correctly by exact match (EM).
2. **Disagreement**: exactly one run scores the example correctly.
3. **Both incorrect**: neither run scores the example correctly.

The default source is the latest experiment under `results/experiments/`. Override `OVERLAP_SOURCE` to point at another experiment directory, an `experiment_report.json`, or a precomputed overlap table.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

try:
    from IPython.display import display
except ModuleNotFoundError:

    def display(obj):
        print(obj)


sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "font.family": "sans-serif",
        "axes.titleweight": "bold",
        "axes.labelweight": "bold",
    }
)


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents, Path("/home/hosak/LEAP")]:
        if (candidate / "results" / "experiments").exists():
            return candidate
    raise FileNotFoundError("Could not locate a repository root containing results/experiments")


def latest_experiment_dir(root):
    experiments_root = root / "results" / "experiments"
    candidates = [p for p in experiments_root.iterdir() if p.is_dir() and (p / "experiment_report.json").exists()]
    if not candidates:
        raise FileNotFoundError(f"No experiment_report.json found under {experiments_root}")
    return max(candidates, key=lambda p: p.stat().st_mtime)


REPO_ROOT = find_repo_root()

# Notebook-level parameters. Keep these near the top when switching comparisons.
# None means: use the latest experiment under results/experiments/.
OVERLAP_SOURCE = None

# Optional run selection. Leave as None to compare the two latest successful runs in OVERLAP_SOURCE.
# Values can be substrings of run_dir/display label, config_key values, or integer positions after sorting by run_dir.
# Display labels omit fields that are not meaningful for that run/configuration.
RUN_A_SELECTOR = None
RUN_B_SELECTOR = None

# EM field used when reading run results.jsonl files. If a row has is_correct/comparison.is_correct,
# those fields take precedence; otherwise this numeric field is thresholded at 1.0.
SCORE_FIELD = "execution_accuracy"

# Optional extractor method for correctness instead of execution EM, e.g. "direct_query", "nl2sql".
# Leave as None for execution-level EM.
EXTRACTOR_METHOD = None

SECTION_ORDER = ["Both correct", "Disagreement", "Both incorrect"]
RUN_COLORS = {"Run A": "#4C72B0", "Run B": "#DD8452"}
SECTION_COLORS = {
    "Both correct": "#55A868",
    "Disagreement": "#C44E52",
    "Both incorrect": "#8172B2",
}

In [2]:
def resolve_overlap_source(source):
    if source is None:
        return latest_experiment_dir(REPO_ROOT)
    source = Path(source).expanduser()
    if not source.is_absolute():
        source = REPO_ROOT / source
    if not source.exists():
        raise FileNotFoundError(f"OVERLAP_SOURCE does not exist: {source}")
    return source


def load_report_from_source(source):
    source = resolve_overlap_source(source)
    if source.is_dir():
        report_path = source / "experiment_report.json"
    elif source.name == "experiment_report.json":
        report_path = source
    else:
        return None, source
    if not report_path.exists():
        raise FileNotFoundError(f"Could not find experiment_report.json at {report_path}")
    report = json.loads(report_path.read_text(encoding="utf-8"))
    return report, report_path.parent


def job_sort_key(job):
    return str(job.get("run_dir", ""))


def describe_job(job):
    parts = [job.get("model"), job.get("strategy")]
    if job.get("strategy") != "direct_query":
        use_constraints = bool(job.get("use_constraints"))
        parts.append("constrained" if use_constraints else "unconstrained")
        if use_constraints and job.get("constraint_backend"):
            parts.append(job.get("constraint_backend"))
        if use_constraints and bool(job.get("use_global_constraints")):
            parts.append("global")
        if job.get("output_format"):
            parts.append(job.get("output_format"))
    if "zero_temp" in str(job.get("config_label", "")):
        parts.append("zero_temp")
    label = " | ".join(str(part) for part in parts if part) or job.get("config_key") or Path(job["run_dir"]).parent.name
    return f"{label} | r{job.get('repeat', '?')}"


def select_job(jobs, selector, default_position):
    sorted_jobs = sorted(jobs, key=job_sort_key)
    if selector is None:
        return sorted_jobs[default_position]
    if isinstance(selector, int):
        return sorted_jobs[selector]
    selector = str(selector)
    matches = [
        job
        for job in sorted_jobs
        if selector in str(job.get("run_dir", ""))
        or selector in str(job.get("config_label", ""))
        or selector == str(job.get("config_key", ""))
    ]
    if len(matches) != 1:
        raise ValueError(f"RUN selector {selector!r} matched {len(matches)} jobs; expected exactly one")
    return matches[0]


def row_correctness(row):
    if EXTRACTOR_METHOD is not None:
        for result in row.get("extractor_results", []):
            if result.get("method") == EXTRACTOR_METHOD:
                return float(result.get("accuracy", 0.0)) == 1.0
        raise KeyError(f"Extractor method {EXTRACTOR_METHOD!r} not found for example {row.get('id')}")
    if "is_correct" in row:
        return bool(row["is_correct"])
    comparison = row.get("comparison") or {}
    if "is_correct" in comparison:
        return bool(comparison["is_correct"])
    return float(row.get(SCORE_FIELD, 0.0)) == 1.0


def load_run_examples(job, run_label):
    run_dir = Path(job["run_dir"])
    rows = []
    with (run_dir / "results.jsonl").open("r", encoding="utf-8") as handle:
        for ordinal, line in enumerate(handle):
            if not line.strip():
                continue
            row = json.loads(line)
            rows.append(
                {
                    "example_id": row.get("example_id") or row.get("request_id") or row.get("id") or str(ordinal),
                    "example_order": ordinal,
                    "question": row.get("question"),
                    f"{run_label}_correct": row_correctness(row),
                }
            )
    return pd.DataFrame(rows)


def load_precomputed_overlap_table(source):
    source = resolve_overlap_source(source)
    if source.suffix == ".csv":
        frame = pd.read_csv(source)
    elif source.suffix in {".json", ".jsonl"}:
        if source.suffix == ".jsonl":
            frame = pd.read_json(source, lines=True)
        else:
            payload = json.loads(source.read_text(encoding="utf-8"))
            frame = pd.DataFrame(payload.get("examples", payload if isinstance(payload, list) else payload.get("rows", [])))
    else:
        raise ValueError(f"Unsupported overlap table format: {source}")
    return frame

In [ ]:
report, resolved_source = load_report_from_source(OVERLAP_SOURCE)

if report is None:
    # Precomputed table path. Expected columns: example_id plus run_a_correct/run_b_correct
    # or Run A_correct/Run B_correct.
    comparison_df = load_precomputed_overlap_table(resolved_source)
    rename_map = {
        "run_a_correct": "Run A_correct",
        "run_b_correct": "Run B_correct",
        "a_correct": "Run A_correct",
        "b_correct": "Run B_correct",
    }
    comparison_df = comparison_df.rename(columns=rename_map)
    if "example_order" not in comparison_df:
        comparison_df["example_order"] = range(len(comparison_df))
    run_a_name = "Run A"
    run_b_name = "Run B"
else:
    jobs = [job for job in report["jobs"] if job.get("status") == "ok" and Path(job.get("run_dir", "") or "").exists()]
    if len(jobs) < 2:
        raise ValueError("Need at least two successful jobs with run directories to compare")
    run_a = select_job(jobs, RUN_A_SELECTOR, -2)
    run_b = select_job(jobs, RUN_B_SELECTOR, -1)
    run_a_name = describe_job(run_a)
    run_b_name = describe_job(run_b)
    run_a_df = load_run_examples(run_a, "Run A")
    run_b_df = load_run_examples(run_b, "Run B")
    comparison_df = run_a_df.merge(
        run_b_df[["example_id", "example_order", "Run B_correct"]],
        on="example_id",
        how="inner",
        suffixes=("", "_b"),
    )
    comparison_df["example_order"] = comparison_df["example_order"].astype(int)

required = {"example_id", "example_order", "Run A_correct", "Run B_correct"}
missing = required - set(comparison_df.columns)
if missing:
    raise ValueError(f"Overlap data is missing required columns: {sorted(missing)}")

comparison_df["Run A_correct"] = comparison_df["Run A_correct"].astype(bool)
comparison_df["Run B_correct"] = comparison_df["Run B_correct"].astype(bool)
comparison_df["section"] = "Disagreement"
comparison_df.loc[comparison_df["Run A_correct"] & comparison_df["Run B_correct"], "section"] = "Both correct"
comparison_df.loc[~comparison_df["Run A_correct"] & ~comparison_df["Run B_correct"], "section"] = "Both incorrect"
comparison_df["winner"] = "Tie"
comparison_df.loc[comparison_df["Run A_correct"] & ~comparison_df["Run B_correct"], "winner"] = "Run A"
comparison_df.loc[~comparison_df["Run A_correct"] & comparison_df["Run B_correct"], "winner"] = "Run B"
comparison_df["section"] = pd.Categorical(comparison_df["section"], categories=SECTION_ORDER, ordered=True)
comparison_df = comparison_df.sort_values(["section", "example_order", "example_id"]).reset_index(drop=True)
comparison_df["section_position"] = comparison_df.groupby("section", observed=False).cumcount()

section_counts = comparison_df["section"].value_counts().reindex(SECTION_ORDER, fill_value=0)
assert int(section_counts.sum()) == len(comparison_df), "Every example should belong to exactly one section"

print(f"Source: {resolved_source}")
print(f"Run A: {run_a_name}")
print(f"Run B: {run_b_name}")
display(section_counts.rename("examples").to_frame())
comparison_df.head()

In [ ]:
def style_section_axis(ax, section, count):
    ax.set_title(f"{section}\n{count:,} examples", pad=12)
    ax.set_xlabel("Example index/order within section")
    ax.set_ylim(-0.6, 1.6)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Run B", "Run A"])
    ax.grid(axis="y", linestyle=(0, (2, 3)), linewidth=0.8, color="#c7c7c7")
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color("#222222")


fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(22, 10),
    sharey=True,
    gridspec_kw={"height_ratios": [max(section_counts[s], 1) for s in SECTION_ORDER]},
)
if len(SECTION_ORDER) == 1:
    axes = [axes]

for ax, section in zip(axes, SECTION_ORDER):
    section_df = comparison_df[comparison_df["section"] == section].copy()
    count = len(section_df)
    ax.axhspan(-0.5, 1.5, color=SECTION_COLORS[section], alpha=0.08, zorder=0)
    if count:
        marker_colors_a = [RUN_COLORS["Run A"] if correct else "white" for correct in section_df["Run A_correct"]]
        marker_colors_b = [RUN_COLORS["Run B"] if correct else "white" for correct in section_df["Run B_correct"]]
        ax.scatter(
            section_df["section_position"],
            [1] * count,
            s=58,
            marker="s",
            facecolors=marker_colors_a,
            edgecolors="black",
            linewidths=0.55,
            label="Run A",
            zorder=3,
        )
        ax.scatter(
            section_df["section_position"],
            [0] * count,
            s=58,
            marker="s",
            facecolors=marker_colors_b,
            edgecolors="black",
            linewidths=0.55,
            label="Run B",
            zorder=3,
        )
    ax.set_xlim(-1, max(count, 1))
    style_section_axis(ax, section, count)

axes[-1].set_xlabel("Example index/order within section")
fig.suptitle("Run A vs Run B exact-match overlap by example", y=0.995, fontsize=22, fontweight="bold")
fig.text(
    0.5,
    0.955,
    f"Run A: {run_a_name}\nRun B: {run_b_name}",
    ha="center",
    va="top",
    fontsize=12,
)

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="s",
        color="none",
        markerfacecolor=RUN_COLORS["Run A"],
        markeredgecolor="black",
        markersize=9,
        label="Correct in Run A",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color="none",
        markerfacecolor=RUN_COLORS["Run B"],
        markeredgecolor="black",
        markersize=9,
        label="Correct in Run B",
    ),
    Line2D([0], [0], marker="s", color="none", markerfacecolor="white", markeredgecolor="black", markersize=9, label="Incorrect"),
    Patch(facecolor=SECTION_COLORS["Both correct"], alpha=0.18, edgecolor="none", label="Both correct section"),
    Patch(facecolor=SECTION_COLORS["Disagreement"], alpha=0.18, edgecolor="none", label="Disagreement section"),
    Patch(facecolor=SECTION_COLORS["Both incorrect"], alpha=0.18, edgecolor="none", label="Both incorrect section"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, -0.005))
plt.tight_layout(rect=[0, 0.055, 1, 0.92])

In [ ]:
# Compact table for decoding disagreement examples.
disagreement_df = comparison_df[comparison_df["section"] == "Disagreement"].copy()
disagreement_df[["example_id", "example_order", "winner", "question"]].head(25)